# Lab 2

---

## Part I — Preparing the Data for Model Training

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedShuffleSplit, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import randint
from pathlib import Path

# Load and recreate train/test split from Lab 1
candidates = [
    Path("datasets/housing/housing.csv"),
    Path("datasets/housing.csv"),
    Path("housing.csv"),
]
housing_file = next((path for path in candidates if path.is_file()), None)

if housing_file is None:
    raise FileNotFoundError("Could not find housing.csv. Check the dataset location.")

housing_raw = pd.read_csv(housing_file)
housing_raw["income_cat"] = pd.cut(
    housing_raw["median_income"],
    bins=[0., 1.5, 3., 4.5, 6., np.inf],
    labels=[1, 2, 3, 4, 5]
)

split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for train_idx, test_idx in split.split(housing_raw, housing_raw["income_cat"]):
    strat_train_set = housing_raw.loc[train_idx].drop("income_cat", axis=1)
    strat_test_set = housing_raw.loc[test_idx].drop("income_cat", axis=1)

# Separate inputs and target
housing = strat_train_set.drop("median_house_value", axis=1)
housing_labels = strat_train_set["median_house_value"].copy()

# Build and apply pipeline
num_attribs = list(housing.drop("ocean_proximity", axis=1))
cat_attribs = ["ocean_proximity"]

num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

full_pipeline = ColumnTransformer([
    ("num", num_pipeline, num_attribs),
    ("cat", OneHotEncoder(), cat_attribs),
])

housing_prepared = full_pipeline.fit_transform(housing)
print("Prepared features shape:", housing_prepared.shape)

FileNotFoundError: [Errno 2] No such file or directory: 'datasets/housing/datasets/housing.csv'

## Part II — Train a Linear Regression Model

In [ ]:
lin_reg = LinearRegression()
lin_reg.fit(housing_prepared, housing_labels)
print("Linear Regression training complete.")

Linear Regression training complete.


## Part III — Generate and Interpret Predictions

In [ ]:
some_data = housing.iloc[:5]
some_labels = housing_labels.iloc[:5]
some_data_prepared = full_pipeline.transform(some_data)

predictions = lin_reg.predict(some_data_prepared)
comp_df = pd.DataFrame({
    "Actual": some_labels.values,
    "Predicted": predictions,
    "Error": some_labels.values - predictions,
    "Absolute Error": np.abs(some_labels.values - predictions)
})
display(comp_df)

,Actual,Predicted,Error,Absolute Error
0,458300.0,270886.932283,187413.067717,187413.067717
1,483800.0,334830.606189,148969.393811,148969.393811
2,101700.0,119856.180089,-18156.180089,18156.180089
3,96100.0,109210.929953,-13110.929953,13110.929953
4,361800.0,305575.658114,56224.341886,56224.341886


## Part IV — Evaluate the Linear Regression Model

In [ ]:
housing_predictions = lin_reg.predict(housing_prepared)
lin_rmse = np.sqrt(mean_squared_error(housing_labels, housing_predictions))
print(f"Linear Regression Training RMSE: ${lin_rmse:.2f}")

Linear Regression Training RMSE: $68232.84


## Part V — Train a Decision Tree Regressor

In [ ]:
tree_reg = DecisionTreeRegressor(random_state=42)
tree_reg.fit(housing_prepared, housing_labels)

tree_predictions = tree_reg.predict(housing_prepared)
tree_rmse = np.sqrt(mean_squared_error(housing_labels, tree_predictions))
print(f"Decision Tree Training RMSE: ${tree_rmse:.2f}")

Decision Tree Training RMSE: $0.00


## Part VI — Investigate Overfitting via Cross-Validation

In [ ]:
scores = cross_val_score(
    tree_reg, housing_prepared, housing_labels,
    scoring="neg_mean_squared_error", cv=10
)
tree_rmse_scores = np.sqrt(-scores)
print(f"Decision Tree CV Mean RMSE: ${tree_rmse_scores.mean():.2f}")
print(f"Decision Tree CV Std Dev:   ${tree_rmse_scores.std():.2f}")

Decision Tree CV Mean RMSE: $68730.84
Decision Tree CV Std Dev:   $1585.36


## Part VII — Compare Models Using Cross-Validation

In [ ]:
lin_scores = cross_val_score(lin_reg, housing_prepared, housing_labels, scoring="neg_mean_squared_error", cv=10)
lin_rmse_scores = np.sqrt(-lin_scores)

forest_reg = RandomForestRegressor(n_estimators=100, random_state=42)
forest_scores = cross_val_score(forest_reg, housing_prepared, housing_labels, scoring="neg_mean_squared_error", cv=10)
forest_rmse_scores = np.sqrt(-forest_scores)

summary_df = pd.DataFrame({
    "Model": ["Linear Regression", "Decision Tree", "Random Forest"],
    "Mean RMSE ($)": [lin_rmse_scores.mean(), tree_rmse_scores.mean(), forest_rmse_scores.mean()],
    "Std. Dev. ($)": [lin_rmse_scores.std(), tree_rmse_scores.std(), forest_rmse_scores.std()]
})
display(summary_df)

KeyboardInterrupt: 

## Part VIII — Fine-Tuning the Model

In [ ]:
# Grid Search
param_grid = [
    {"n_estimators": [3, 10, 30], "max_features": [2, 4, 6, 8]},
    {"bootstrap": [False], "n_estimators": [3, 10], "max_features": [2, 3, 4]}
]
forest_reg = RandomForestRegressor(random_state=42)
grid_search = GridSearchCV(forest_reg, param_grid, cv=5, scoring="neg_mean_squared_error", return_train_score=True)
grid_search.fit(housing_prepared, housing_labels)

print("Grid Search Best Params:", grid_search.best_params_)
print(f"Grid Search Best RMSE: ${np.sqrt(-grid_search.best_score_):.2f}")

# Randomized Search
param_distribs = {
    "n_estimators": randint(low=1, high=200),
    "max_features": randint(low=1, high=8)
}
rnd_search = RandomizedSearchCV(forest_reg, param_distributions=param_distribs, n_iter=10, cv=5, scoring="neg_mean_squared_error", random_state=42)
rnd_search.fit(housing_prepared, housing_labels)

print("Randomized Search Best Params:", rnd_search.best_params_)
print(f"Randomized Search Best RMSE: ${np.sqrt(-rnd_search.best_score_):.2f}")

Grid Search Best Params: {'max_features': 6, 'n_estimators': 30}
Grid Search Best RMSE: $49323.24
Randomized Search Best Params: {'max_features': 7, 'n_estimators': 180}
Randomized Search Best RMSE: $48704.58


## Part IX — Analyze the Best Model and Model Error

In [ ]:
final_model = grid_search.best_estimator_
feature_importances = final_model.feature_importances_
feature_names = full_pipeline.get_feature_names_out()

imp_df = pd.DataFrame({"Feature": feature_names, "Importance": feature_importances}).sort_values(by="Importance", ascending=False)
display(imp_df)

,Feature,Importance
7,num__median_income,0.404648
9,cat__ocean_proximity_INLAND,0.162404
0,num__longitude,0.119607
1,num__latitude,0.107533
2,num__housing_median_age,0.044747
5,num__population,0.044152
3,num__total_rooms,0.034784
4,num__total_bedrooms,0.030468
6,num__households,0.027818
8,cat__ocean_proximity_<1H OCEAN,0.013902


In [ ]:
# Error Analysis
train_preds = final_model.predict(housing_prepared)
errors = housing_labels.values - train_preds
err_df = pd.DataFrame({"Actual": housing_labels.values, "Predicted": train_preds, "Error": errors, "Abs Error": np.abs(errors)})
display(err_df.sort_values(by="Abs Error", ascending=False).head(10))

,Actual,Predicted,Error,Abs Error
1238,475000.0,295526.666667,179473.333333,179473.333333
13063,500001.0,328230.566667,171770.433333,171770.433333
14223,500001.0,334683.766667,165317.233333,165317.233333
468,500001.0,337083.866667,162917.133333,162917.133333
1345,500001.0,348020.533333,151980.466667,151980.466667
15137,450000.0,302136.666667,147863.333333,147863.333333
5985,450000.0,303979.966667,146020.033333,146020.033333
3270,500001.0,356893.900000,143107.100000,143107.100000
15129,500001.0,363793.866667,136207.133333,136207.133333
12597,500001.0,364060.500000,135940.500000,135940.500000


## Part X — Final Evaluation on the Test Set

In [ ]:
X_test = strat_test_set.drop("median_house_value", axis=1)
y_test = strat_test_set["median_house_value"].copy()

X_test_prepared = full_pipeline.transform(X_test)
final_preds = final_model.predict(X_test_prepared)

final_rmse = np.sqrt(mean_squared_error(y_test, final_preds))
final_mae = mean_absolute_error(y_test, final_preds)

print(f"Final Test RMSE: ${final_rmse:.2f}")
print(f"Final Test MAE:  ${final_mae:.2f}")

Final Test RMSE: $49763.37
Final Test MAE:  $32476.79


## Part XI — Run a Complete Prediction Pipeline

In [ ]:
full_prediction_pipeline = Pipeline([
    ("preparation", full_pipeline),
    ("model", final_model)
])

pipeline_preds = full_prediction_pipeline.predict(X_test)
print("Complete pipeline works. First 5 test predictions:\n", pipeline_preds[:5])

Complete pipeline works. First 5 test predictions:
 [406183.5        199493.33333333 338906.8        327370.03333333
 277243.46666667]


### Task 11.1 — Understand the Complete Pipeline

Combines imputation, scaling, categorical encoding, and inference into a single step, preventing data leakage during inference.